In [1]:
!pip install torch-geometric
import os, gc, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from transformers import AutoConfig, AutoModel, AutoTokenizer

KAGGLE_INPUT = Path('/kaggle/input')
OUT_DIR      = Path('/kaggle/working/adv_ext')
OUT_DIR.mkdir(exist_ok=True)

SEEDS    = [42, 1337, 2024]
LABEL_COL = 'label'
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def _rglob_first(root, pattern):
    hits = list(root.rglob(pattern))
    return hits[0] if hits else None

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
Device: cuda


In [2]:
# Try test_adversarial.parquet first (saved by NB02)
adv_path = _rglob_first(KAGGLE_INPUT, 'test_adversarial.parquet')

if adv_path is None:
    # Fall back: filter obfusc_* from the full corpus parquet
    corpus_path = _rglob_first(KAGGLE_INPUT, 'powershell_corpus_v3.parquet')
    assert corpus_path is not None, (
        "Neither test_adversarial.parquet nor powershell_corpus_v3.parquet found.\n"
        "Attach 03-splits-v3-dataset or 02-corpus-v4-dataset."
    )
    df_corp = pd.read_parquet(corpus_path, columns=[LABEL_COL, 'source', 'script_text'])
    ADV_DF  = df_corp[df_corp['source'].str.startswith('obfusc_')].reset_index(drop=True)
    print(f"Built adversarial holdout from corpus: {len(ADV_DF):,} scripts")
else:
    ADV_DF = pd.read_parquet(adv_path)
    print(f"Loaded test_adversarial.parquet: {len(ADV_DF):,} scripts")

# All must be malicious
assert (ADV_DF[LABEL_COL] == 1).all(), "Adversarial holdout should be all malicious"
print(f"Obfuscation types:\n{ADV_DF['source'].value_counts().to_string()}")

adv_texts  = ADV_DF['script_text'].fillna('').tolist()
adv_labels = ADV_DF[LABEL_COL].astype(np.int32).values
obfusc_types = sorted(ADV_DF['source'].unique())

Loaded test_adversarial.parquet: 9,164 scripts
Obfuscation types:
source
obfusc_compress    2299
obfusc_encode      2283
obfusc_string      1287
obfusc_format      1158
obfusc_concat      1098
obfusc_alias       1039


In [3]:
# ── CodeBERT model definition (must match NB04 exactly) ─────────────────────
MODEL_NAME   = 'microsoft/codebert-base'
MAX_SEQ_LEN  = 512
STRIDE       = 128    # NB04 uses stride=128, not 256
MAX_WINDOWS  = 8

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)

class CodeBertClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        self.config     = AutoConfig.from_pretrained(model_name)
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden          = self.config.hidden_size    # 768
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        out        = self.encoder(input_ids=input_ids,
                                  attention_mask=attention_mask, return_dict=True)
        cls_hidden = out.last_hidden_state[:, 0, :]
        x          = self.dropout(cls_hidden)
        logits     = self.classifier(x).squeeze(-1)   # [B]
        return torch.sigmoid(logits)

# 👇 OUTDENTED: Now a standalone function instead of a class method
def cb_sliding_window_probs(texts, model, batch_size=64):
    """Max-pool sliding-window inference. Returns numpy array of probs."""
    model.eval()
    all_probs = []
    
    with torch.no_grad():
        # Processing one script at a time. The "batch" becomes the chunks of the script.
        for text in texts:
            # 1. Tokenize into overlapping chunks
            inputs = TOKENIZER(
                text,
                truncation=True,
                max_length=MAX_SEQ_LEN, 
                stride=STRIDE,          
                return_overflowing_tokens=True,
                padding="max_length",
                return_tensors="pt"
            )
            
            # The tokenizer returns a mapping key that the model doesn't accept
            inputs.pop("overflow_to_sample_mapping", None)
            
            # 2. Cap the maximum number of windows to avoid VRAM exhaustion on massive scripts
            if inputs['input_ids'].shape[0] > MAX_WINDOWS:
                inputs['input_ids'] = inputs['input_ids'][:MAX_WINDOWS]
                inputs['attention_mask'] = inputs['attention_mask'][:MAX_WINDOWS]
                
            # Move tensors to the active device
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # 3. Forward pass for all chunks of this single script
            # chunk_probs shape: [num_chunks]
            chunk_probs = model(inputs['input_ids'], inputs['attention_mask'])
            
            # 4. Max-pooling: Take the highest probability among all chunks
            script_prob = chunk_probs.max().item()
            all_probs.append(script_prob)
            
    return np.array(all_probs)


# ── Locate CodeBERT checkpoint directory ─────────────────────────────────────
CB_MODELS_BASE = None
for cand in KAGGLE_INPUT.rglob('codebert_seed42'):
    CB_MODELS_BASE = cand.parent
    break

if CB_MODELS_BASE is None:
    print("[SKIP] CodeBERT checkpoints not found. Attach lotl-nlp-outputs dataset.")
    cb_adv_rows = []
else:
    print(f"CodeBERT checkpoints found at: {CB_MODELS_BASE}")
    cb_adv_rows = []

    for seed in SEEDS:
        enc_dir = CB_MODELS_BASE / f'codebert_seed{seed}' / 'encoder'
        head_pt = CB_MODELS_BASE / f'codebert_seed{seed}' / 'classifier_head.pt'

        if not enc_dir.exists() or not head_pt.exists():
            print(f"  [SKIP] seed={seed}: checkpoint files missing")
            continue

        print(f"\n  Loading CodeBERT seed={seed} ...")
        t0 = time.time()
        
        model = CodeBertClassifier(model_name=str(enc_dir))
        model.classifier.load_state_dict(
            torch.load(head_pt, map_location=device)
        )
        model = model.to(device)
        model.eval()

        probs = cb_sliding_window_probs(adv_texts, model)
        overall_tpr = float((probs >= 0.5).mean())
        tpr_by_type = {
            t: round(float((probs[ADV_DF['source'].values == t] >= 0.5).mean()), 4)
            for t in obfusc_types
        }
        cb_adv_rows.append({
            'seed': seed, 'tpr': round(overall_tpr, 4),
            'tpr_by_type': tpr_by_type
        })
        print(f"    seed={seed}  overall TPR={overall_tpr:.4f}  "
              f"({(time.time()-t0)/60:.1f} min)")

        del model
        gc.collect()
        torch.cuda.empty_cache()

    if cb_adv_rows:
        tprs = [r['tpr'] for r in cb_adv_rows]
        print(f"\nCodeBERT adversarial TPR: {np.mean(tprs):.4f} ± {np.std(tprs):.4f}")
        for t in obfusc_types:
            vals = [r['tpr_by_type'].get(t) for r in cb_adv_rows]
            print(f"  {t:<28}  {np.mean(vals):.4f} ± {np.std(vals):.4f}")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

    seed=42  overall TPR=0.9975  (50.6 min)

  Loading CodeBERT seed=1337 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    seed=1337  overall TPR=0.9937  (51.2 min)

  Loading CodeBERT seed=2024 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    seed=2024  overall TPR=0.9931  (52.4 min)

CodeBERT adversarial TPR: 0.9948 ± 0.0019
  obfusc_alias                  0.9849 ± 0.0073
  obfusc_compress               1.0000 ± 0.0000
  obfusc_concat                 0.9897 ± 0.0055
  obfusc_encode                 1.0000 ± 0.0000
  obfusc_format                 0.9902 ± 0.0018
  obfusc_string                 0.9925 ± 0.0020


In [4]:
# ── GATv2 architecture (must match NB05 exactly) ─────────────────────────────
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Batch
import torch.nn.functional as F
import torch
import torch.nn as nn
import numpy as np
import time
import gc

GAT_HIDDEN_DIM      = 128
GAT_HEADS           = 4
GAT_NUM_LAYERS      = 2
GAT_DROPOUT         = 0.1       # v3 value
GAT_NEG_SLOPE       = 0.2
GRAPH_EMB_DIM       = 2 * GAT_HIDDEN_DIM       # 256
NODE_TYPE_VOCAB_CAP = 200
TOKEN_HASH_DIM      = 32
NODE_FEAT_DIM       = NODE_TYPE_VOCAB_CAP + TOKEN_HASH_DIM   # 232
MAX_NODES           = 800


class GATv2Classifier(nn.Module):
    def __init__(self, in_dim=NODE_FEAT_DIM, hidden=GAT_HIDDEN_DIM,
                 heads=GAT_HEADS, num_layers=GAT_NUM_LAYERS, dropout=GAT_DROPOUT):
        super().__init__()
        assert num_layers == 2
        self.dropout_p   = dropout
        
        self.gat1        = GATv2Conv(in_dim, hidden, heads=heads, concat=True,
                                     dropout=dropout, negative_slope=GAT_NEG_SLOPE,
                                     add_self_loops=True)
        self.gat2        = GATv2Conv(hidden*heads, hidden, heads=heads, concat=False,
                                     dropout=dropout, negative_slope=GAT_NEG_SLOPE,
                                     add_self_loops=True)
        
        self.skip_proj    = nn.Linear(hidden*heads, hidden)
        self.head_dropout = nn.Dropout(dropout)
        self.head         = nn.Linear(GRAPH_EMB_DIM, 1)

    def encode(self, x, edge_index, batch):
        x  = x.float()
        h1 = F.elu(self.gat1(x, edge_index))
        h1 = F.dropout(h1, p=self.dropout_p, training=self.training)
        
        h2 = self.gat2(h1, edge_index) + self.skip_proj(h1)
        h  = F.elu(h2)
        h  = F.dropout(h, p=self.dropout_p, training=self.training)
        
        return torch.cat([global_mean_pool(h, batch), global_max_pool(h, batch)], dim=-1)

    def forward(self, x, edge_index, batch, labels=None, pos_weight=None, return_embedding=False):
        g = self.encode(x, edge_index, batch)
        logits = self.head(self.head_dropout(g)).squeeze(-1)
        
        # Match NB05 return dictionary format
        if return_embedding:
            return {"logits": logits, "embedding": g}
        return {"logits": logits}


# ── Graph construction from raw PowerShell text ──────────────────────────────
from torch_geometric.data import Data
import hashlib
import re

_PS_TOKEN_RE = re.compile(r'[A-Za-z_\-][\w\-]*|"[^"]*"|\'[^\']*\'|\$\w+|[\(\)\{\}\[\]\|;]')

def _hash_token_to_vec(token, dim=TOKEN_HASH_DIM):
    """Matches NB05's exact sha256 + tanh implementation"""
    if not token:
        return np.zeros(dim, dtype=np.float32)
    h = hashlib.sha256(token.encode("utf-8", errors="replace")).digest()
    needed = dim * 4
    if len(h) < needed:
        h = (h * (needed // len(h) + 1))[:needed]
    arr = np.frombuffer(h[:needed], dtype=np.int32).astype(np.float32)
    return np.tanh(arr / (2**24))

def text_to_graph(script_text, max_nodes=MAX_NODES):
    """
    Convert a PowerShell script string to a PyG Data object.
    Uses token-level linear chain graph as a fallback AST approximation.
    """
    tokens = _PS_TOKEN_RE.findall(script_text[:8192])  # cap at 8192 chars
    if not tokens:
        tokens = ['<empty>']
    tokens = tokens[:max_nodes]
    n = len(tokens)
    
    # Build type_ids: use simple hash-based category (0–199)
    # (Note: For absolute exact evaluation, load type_vocab.json from NB05)
    type_ids = np.array(
        [int(hashlib.md5(t.lower().encode()).hexdigest(), 16) % NODE_TYPE_VOCAB_CAP
         for t in tokens],
        dtype=np.int32
    )
    
    # Hash embeddings per token
    hash_cache = np.zeros((n, TOKEN_HASH_DIM), dtype=np.float16)
    for i, t in enumerate(tokens):
        hash_cache[i] = _hash_token_to_vec(t).astype(np.float16)
    
    # Feature matrix: one-hot type + hash
    x = np.zeros((n, NODE_FEAT_DIM), dtype=np.float16)
    x[np.arange(n), type_ids] = 1.0
    x[:, NODE_TYPE_VOCAB_CAP:] = hash_cache
    
    # Linear chain edges + reverse edges (undirected)
    if n > 1:
        src = np.arange(n - 1, dtype=np.int64)
        dst = np.arange(1, n, dtype=np.int64)
        edges = np.stack([
            np.concatenate([src, dst]),
            np.concatenate([dst, src])
        ], axis=0)
    else:
        edges = np.array([[0], [0]], dtype=np.int64)
    
    return Data(
        x          = torch.from_numpy(x),
        edge_index = torch.from_numpy(edges),
        y          = torch.tensor([1])   # all adversarial scripts are malicious
    )


def gat_infer_probs(graphs, model, batch_size=64):
    """Batch inference on a list of PyG Data objects."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(graphs), batch_size):
            batch = Batch.from_data_list(graphs[i : i + batch_size]).to(device)
            
            # Model returns a dict, and we use BCE so we need sigmoid
            out = model(batch.x, batch.edge_index, batch.batch)
            logits = out["logits"]
            
            # Convert logits to probabilities
            probs = torch.sigmoid(logits.float()).cpu().numpy()
            
            # Handle potential 0D array if batch_size ends up being 1
            if probs.ndim == 0:
                all_probs.append(float(probs))
            else:
                all_probs.extend(probs.tolist())
                
    return np.array(all_probs, dtype=np.float32)


# ── Build graphs once (reused across all 3 seeds) ────────────────────────────
print(f"Building graphs for {len(adv_texts):,} adversarial scripts ...")
t0 = time.time()
adv_graphs = [text_to_graph(t) for t in adv_texts]
print(f"  Done in {time.time()-t0:.1f}s  "
      f"(avg nodes: {np.mean([g.x.shape[0] for g in adv_graphs]):.1f})")

# ── Locate GATv2 checkpoint directory ────────────────────────────────────────
GAT_MODELS_BASE = None
for cand in KAGGLE_INPUT.rglob('gat_seed42'):
    GAT_MODELS_BASE = cand.parent
    break

if GAT_MODELS_BASE is None:
    print("[SKIP] GATv2 checkpoints not found. Attach lotl-gat-v3-outputs dataset.")
    gat_adv_rows = []
else:
    print(f"GATv2 checkpoints found at: {GAT_MODELS_BASE}")
    gat_adv_rows = []

    for seed in SEEDS:
        ckpt_pt = GAT_MODELS_BASE / f'gat_seed{seed}' / 'model.pt'
        if not ckpt_pt.exists():
            print(f"  [SKIP] seed={seed}: model.pt not found at {ckpt_pt}")
            continue

        print(f"\n  Loading GATv2 seed={seed} ...")
        t0 = time.time()
        
        model = GATv2Classifier().to(device)
        model.load_state_dict(
            torch.load(ckpt_pt, map_location=device, weights_only=True)
        )
        model.eval()

        probs = gat_infer_probs(adv_graphs, model)
        overall_tpr = float((probs >= 0.5).mean())
        tpr_by_type = {
            t: round(float((probs[ADV_DF['source'].values == t] >= 0.5).mean()), 4)
            for t in obfusc_types
        }
        gat_adv_rows.append({
            'seed': seed, 'tpr': round(overall_tpr, 4),
            'tpr_by_type': tpr_by_type
        })
        print(f"    seed={seed}  overall TPR={overall_tpr:.4f}  "
              f"({time.time()-t0:.1f}s)")

        del model
        gc.collect()
        torch.cuda.empty_cache()

    if gat_adv_rows:
        tprs = [r['tpr'] for r in gat_adv_rows]
        print(f"\nGATv2 adversarial TPR: {np.mean(tprs):.4f} ± {np.std(tprs):.4f}")
        for t in obfusc_types:
            vals = [r['tpr_by_type'].get(t) for r in gat_adv_rows]
            print(f"  {t:<28}  {np.mean(vals):.4f} ± {np.std(vals):.4f}")

Building graphs for 9,164 adversarial scripts ...
  Done in 23.1s  (avg nodes: 245.1)
GATv2 checkpoints found at: /kaggle/input/datasets/onkarkmane1501/lotl-gat-v3-outputs/gat_v3/models

  Loading GATv2 seed=42 ...
    seed=42  overall TPR=0.8335  (4.9s)

  Loading GATv2 seed=1337 ...
    seed=1337  overall TPR=0.5571  (4.3s)

  Loading GATv2 seed=2024 ...
    seed=2024  overall TPR=0.8891  (4.4s)

GATv2 adversarial TPR: 0.7599 ± 0.1452
  obfusc_alias                  0.8191 ± 0.0465
  obfusc_compress               0.6993 ± 0.3882
  obfusc_concat                 0.8042 ± 0.0587
  obfusc_encode                 0.9948 ± 0.0019
  obfusc_format                 0.5947 ± 0.1387
  obfusc_string                 0.5146 ± 0.1491


In [5]:
result = {
    'xgboost': None,           # to be filled from NB07 adversarial_table5.json
    'codebert': cb_adv_rows,
    'gatv2': gat_adv_rows,
    'obfusc_types': obfusc_types,
    'n_scripts': len(adv_texts),
}

with open(OUT_DIR / 'adversarial_ext_results.json', 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved: {OUT_DIR / 'adversarial_ext_results.json'}")

# Print merged summary table
print('\n=== Extended Table 5 — Adversarial Evaluation (all 3 branches) ===')
print(f'N={len(adv_texts):,} scripts (all malicious)\n')
print(f"{'Obfuscation type':<28}  {'XGB (NB07)':>12}  {'CodeBERT':>12}  {'GATv2':>12}")
print('-' * 70)

for t in obfusc_types:
    cb_vals  = [r['tpr_by_type'].get(t, float('nan')) for r in cb_adv_rows]  if cb_adv_rows  else []
    gat_vals = [r['tpr_by_type'].get(t, float('nan')) for r in gat_adv_rows] if gat_adv_rows else []
    cb_str   = f"{np.mean(cb_vals):.4f}"  if cb_vals  else "  N/A "
    gat_str  = f"{np.mean(gat_vals):.4f}" if gat_vals else "  N/A "
    print(f"  {t:<26}  {'(see NB07)':>12}  {cb_str:>12}  {gat_str:>12}")

print()
if cb_adv_rows:
    cb_overall = np.mean([r['tpr'] for r in cb_adv_rows])
    cb_std     = np.std( [r['tpr'] for r in cb_adv_rows])
    print(f"  {'Overall TPR':<26}  {'(see NB07)':>12}  {cb_overall:.4f}±{cb_std:.4f}  ", end='')
if gat_adv_rows:
    gat_overall = np.mean([r['tpr'] for r in gat_adv_rows])
    gat_std     = np.std( [r['tpr'] for r in gat_adv_rows])
    print(f"{gat_overall:.4f}±{gat_std:.4f}")

Saved: /kaggle/working/adv_ext/adversarial_ext_results.json

=== Extended Table 5 — Adversarial Evaluation (all 3 branches) ===
N=9,164 scripts (all malicious)

Obfuscation type                XGB (NB07)      CodeBERT         GATv2
----------------------------------------------------------------------
  obfusc_alias                  (see NB07)        0.9849        0.8191
  obfusc_compress               (see NB07)        1.0000        0.6993
  obfusc_concat                 (see NB07)        0.9897        0.8042
  obfusc_encode                 (see NB07)        1.0000        0.9948
  obfusc_format                 (see NB07)        0.9902        0.5947
  obfusc_string                 (see NB07)        0.9925        0.5146

  Overall TPR                   (see NB07)  0.9948±0.0019  0.7599±0.1452
